# Baseline models

In [5]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


from src.data import CraigslistPreprocessor


In [6]:
data_path = Path("data/interim/train_filtered.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Не найден файл: {data_path}")

df = pd.read_csv(data_path, index_col=0)

y = df["price"].copy()
X = df.drop(columns=["price"]).copy()

print(f"Data path: {data_path}")
print(f"X shape: {X.shape}, y shape: {y.shape}")


Data path: data\interim\train_filtered.csv
X shape: (182190, 18), y shape: (182190,)


In [7]:
def categorical_without_description(X_df: pd.DataFrame) -> list[str]:
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    return [col for col in cat_cols if col != "description"]


def numeric_columns(X_df: pd.DataFrame) -> list[str]:
    return X_df.select_dtypes(exclude=["object", "category"]).columns.tolist()


def make_pipeline(model):
    feature_encoder = ColumnTransformer(
        transformers=[
            (
                "description_tfidf",
                TfidfVectorizer(
                    max_features=20000,
                    ngram_range=(1, 2),
                    min_df=5,
                ),
                "description",
            ),
            (
                "categorical_ohe",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_without_description,
            ),
            (
                "numeric",
                "passthrough",
                numeric_columns,
            ),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("preprocess", CraigslistPreprocessor()),
            ("encode", feature_encoder),
            ("model", model),
        ]
    )


In [8]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"mae": make_scorer(mean_absolute_error, greater_is_better=False)}

models = {
    "DummyRegressor": DummyRegressor(strategy="median"),
    "LinearRegression": LinearRegression(),
}

rows = []
for name, model in models.items():
    pipe = make_pipeline(model)
    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    mae_scores = -scores["test_mae"]
    rows.append(
        {
            "model": name,
            "mae_mean": mae_scores.mean(),
            "mae_std": mae_scores.std(),
            "mae_folds": mae_scores,
        }
    )

results = pd.DataFrame(rows).sort_values("mae_mean").reset_index(drop=True)
results


c:\Users\VLAD\anaconda3\envs\HW2\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\VLAD\anaconda3\envs\HW2\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\VLAD\AppData\Local\Temp\ipykernel_7392\756846667.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

,model,mae_mean,mae_std,mae_folds
0,LinearRegression,6192.721167,40.138301,"[6208.781305898925, 6153.542573294098, 6136.84..."
1,DummyRegressor,10657.999704,46.990097,"[10739.963938745266, 10614.914375102915, 10613..."


In [9]:
best = results.loc[0]
print(f"Best baseline: {best['model']}")
print(f"MAE: {best['mae_mean']:.2f} +/- {best['mae_std']:.2f}")


Best baseline: LinearRegression
MAE: 6192.72 +/- 40.14
